## What makes a basketball team win?

The first time I watched Shai Gilgeous-Alexander play on TV, I was treated to the statistic that it was his `20`th consecutive game in the `2025-2026` season in which he had scored 20+ points (this happened against the Suns on November 28, 2025).


What an insane stat! Most players would be pretty happy to have a handful of `20`-point matches throughout the season; a streak like that could be career-changing for the average player.


Yet, basketball is a team sport. During the `2025-2026` playoffs, despite SGA posting playoff averages of `27.6` points and `7.9` assists per game, OKC were not able to repeat their stellar, championship-winning performance from their previous season.


This led me to ponder the initial 2-pronged question: 
`1. During the playoffs, what box score factors influence the probability of a win?`
`2. Do these factors differ for the regular season?`

To redefine the question for code, let's define a statistical backtest with parameters.

The goal of this backtest is to use box score metrics to infer the historical probability of a win.

---

`Sample size: 496`

I'm using data from the 2021-2026 season, which gives me `82` games/season * `6` seasons.

In [18]:
from nba_api.stats.endpoints import leaguedashteamstats
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
import time

seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]

# Building a dataset with all seasons data
dfs_base = []

for season in seasons:
    print(f"Fetching {season}...")

    data = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star="Regular Season",
        per_mode_detailed="PerGame",
        measure_type_detailed_defense="Base",
        timeout=60
    )

    df_season = data.get_data_frames()[0]
    df_season["SEASON"] = season
    dfs_base.append(df_season)
    time.sleep(1)

print("Fetches completed successfully!\n")

df_all = pd.concat(dfs_base, ignore_index=True)

print("Printing dataset header rows:\n")

print(df_all.head())

Fetching 2021-22...
Fetching 2022-23...
Fetching 2023-24...
Fetching 2024-25...
Fetching 2025-26...
Fetches completed successfully!

Printing dataset header rows:

      TEAM_ID          TEAM_NAME  GP   W   L  W_PCT   MIN   FGM   FGA  FG_PCT  \
0  1610612737      Atlanta Hawks  82  43  39  0.524  48.1  41.5  88.3   0.470   
1  1610612738     Boston Celtics  82  51  31  0.622  48.5  40.7  87.4   0.466   
2  1610612751      Brooklyn Nets  82  44  38  0.537  48.2  42.0  88.4   0.475   
3  1610612766  Charlotte Hornets  82  43  39  0.524  48.5  42.8  91.4   0.468   
4  1610612741      Chicago Bulls  82  46  36  0.561  48.1  41.7  86.9   0.480   

   ...  AST_RANK  TOV_RANK  STL_RANK  BLK_RANK  BLKA_RANK  PF_RANK  PFD_RANK  \
0  ...        15         1        22        23         10        7         6   
1  ...        14        13        19         2         11        5        20   
2  ...        10        17        24         5         21       22        16   
3  ...         1        10   

Noisy dataset! Let's define only what we need.

In [ ]:
# Target
target = "W_PCT"

# Columns to remove from features
drop_exact = {
    # removing raw scoring totals, game metadata, any vector that affects the outcomes directly
    "PTS",
    "FGM",
    "FGA",
    "FTM",
    "FTA",
    "OREB",
    "DREB",
    "MIN",
    "GP",
    "G",
    "W_PCT",
    "W",
    "L",
    "TEAM_ID",
    "SEASON",
    "PLUS_MINUS",
}

# Not necessary for a regression to know the ranks
drop_patterns = [
    "_RANK"
]

cols_to_drop = [
    col for col in df_all.columns
    if (
        col in drop_exact
        or any(pat in col for pat in drop_patterns)
    )
]

y = df_all[target]

# Ensure only numeric features are included in the model
X = df_all.drop(columns=cols_to_drop).select_dtypes(include="number")

# Building a combined modelling dataframe for fluency
df_model = pd.concat([y, X], axis=1).dropna()

In [ ]:
# -----------------------------
# EXTRACTING TARGET AND FEATURES
# -----------------------------

y = df_model["W_PCT"]
X = df_model.drop(columns=["W_PCT"])

# -----------------------------
# SCALE FEATURES
# An important step in the regression.
# This normalizes the features to ensure that comparisons are fair.
# -----------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# ELASTIC NET REGRESSION
# -----------------------------

model = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    cv=5,
    random_state=42,
    max_iter=100000
)

model.fit(X_scaled, y)

# -----------------------------
# FEATURE IMPORTANCE
# -----------------------------

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coef_df["Abs_Coeff"] = coef_df["Coefficient"].abs()

important_features = (
    coef_df[
        (coef_df["Coefficient"] != 0)
        & (coef_df["Abs_Coeff"] >= 0.01)
    ]
    .sort_values("Abs_Coeff", ascending=False)
)

important_features["Coefficient"] = important_features["Coefficient"].round(3)

print(
    important_features[
        ["Feature", "Coefficient"]
    ]
)

    Feature  Coefficient
5       REB        0.054
0    FG_PCT        0.048
8       STL        0.041
7       TOV       -0.040
10     BLKA       -0.034
3   FG3_PCT        0.033
12      PFD        0.022
1      FG3M        0.015
6       AST       -0.015
4    FT_PCT        0.013


Ignoring assists, I can build a model from here.

In [9]:
features = [
    "FG_PCT",
    "FG3M",
    "FG3_PCT",
    "FT_PCT",
    "REB",
    "TOV",
    "STL",
    "BLKA",
    "PFD"
]

new_team = pd.DataFrame([{
    "FG_PCT": 0.485,
    "FG3M": 13.2,
    "FG3A": 35.5,
    "FG3_PCT": 0.372,
    "FT_PCT": 0.791,
    "REB": 44.1,
    "AST": 26.4,
    "TOV": 12.8,
    "STL": 7.9,
    "BLK": 5.1,
    "BLKA": 4.3,
    "PF": 18.2,
    "PFD": 19.7
}])

# force same column order as training
new_team = new_team[X.columns]

new_team_scaled = scaler.transform(new_team)

predicted_w_pct = model.predict(new_team_scaled)

print(predicted_w_pct)

[0.67371795]
